### Setting up Websocket server

In [ ]:
"""
# "headless" = no GUI dependencies
pip install websockets opencv-python-headless numpy --break-system-packages

# ensure Pi's firewall allows traffic on port 8765
sudo ufw allow 8765
"""

### On Raspberry Pi

In [ ]:
import asyncio
import websockets
import subprocess
import base64

async def handler(websocket, path):
    """
    This function is called for each new WebSocket client.
    We'll wait for messages from the client.
    If a message == 'capture', we run libcamera-jpeg.
    """
    print("Client connected!")
    output_file = "test.jpg"

    try:
        async for message in websocket:
            print(f"Received message: {message}")

            if message.lower() == "capture":
                # 1) Run libcamera-jpeg to take a photo
                try:
                    subprocess.run(["libcamera-jpeg", "n", "-t", "1", "-o", output_file], check=True)
                    print(f"Photo captured and saved as {output_file}")

                    # 2a) Option A: Send back a simple success text
                    # await websocket.send("Photo captured successfully!")

                    # 2b) Option B: Send back the actual image as base64
                    with open(output_file, "rb") as f:
                        img_data = f.read()
                    encoded = base64.b64encode(img_data).decode('utf-8')
                    # Send the base64 string
                    await websocket.send(encoded)
                    print("Sent base64-encoded image back to client.")

                except subprocess.CalledProcessError as e:
                    error_msg = f"Failed to capture photo: {e}"
                    print(error_msg)
                    await websocket.send(error_msg)
            else:
                # For any other message, we can just echo or ignore
                await websocket.send("Unknown command. Try 'capture'.")
    except websockets.exceptions.ConnectionClosed:
        print("Client disconnected.")

async def main():
    # Start the WebSocket server on all interfaces (0.0.0.0) at port 8765
    async with websockets.serve(handler, "0.0.0.0", 8765):
        print("WebSocket server started on port 8765. Waiting for clients...")
        await asyncio.Future()  # Run forever

if __name__ == "__main__":
    asyncio.run(main())